In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,recall_score,cohen_kappa_score,accuracy_score
from scipy.io import loadmat
import os
%matplotlib inline

In [ ]:
##hypeperameters and experimental settings
RANDOM_SEED=666
DATASET = 'IP'    ## PU  IP  SA  
TRAIN_RATE = 0.05  ## ratio of training data
VAL_RATE = 0.05    ## ratio of valuating data
EPOCH = 200    ##number of teacher epoch
VAL_EPOCH = 1  ##interval of valuation
LR = 0.001    ##learning rate
WEIGHT_DECAY = 1e-3
BATCH_SIZE = 64
N_PCA = 64
DEVICE = 0  ##-1:CPU  0:cuda 0
PATCH_SIZE = 13
NHEADS = 8
FEEDFORWARD_FACTOR = 4
NUM_LAYERS = 3
SAVE_PATH = f"results\\{DATASET}"
if not os.path.isdir(SAVE_PATH):
    os.mkdir(SAVE_PATH)

In [ ]:
## Set random seed for reproduction
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
torch.use_deterministic_algorithms(True)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED) # if you are using multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def loadData(name):
    data_path = os.path.join(os.getcwd(),'dataset')
    if name == 'IP':
        data = loadmat(os.path.join(data_path, 'IndianPines\\Indian_pines_corrected.mat'))['indian_pines_corrected']
        labels = loadmat(os.path.join(data_path, 'IndianPines\\Indian_pines_gt.mat'))['indian_pines_gt']
        class_name = [ "Alfalfa", "Corn-notill", "Corn-mintill","Corn", "Grass-pasture", 
                       "Grass-trees","Grass-pasture-mowed", "Hay-windrowed", "Oats","Soybean-notill", "Soybean-mintill", "Soybean-clean","Wheat", "Woods", "Buildings-Grass-Trees-Drives","Stone-Steel-Towers"]
    elif name == 'SA':
        data = loadmat(os.path.join(data_path, 'Salinas\\Salinas_corrected.mat'))['salinas_corrected']
        labels = loadmat(os.path.join(data_path, 'Salinas\\Salinas_gt.mat'))['salinas_gt']
        class_name = ['Brocoli_green_weeds_1','Brocoli_green_weeds_2','Fallow',
                        'Fallow_rough_plow','Fallow_smooth','Stubble','Celery','Grapes_untrained','Soil_vinyard_develop','Corn_senesced_green','Lettuce_romaine_4wk','Lettuce_romaine_5wk','Lettuce_romaine_6wk','Lettuce_romaine_7wk','Vinyard_untrained','Vinyard_vertical']
    elif name == 'LK':
        data = loadmat(os.path.join(data_path, 'WHU_Hi_LongKou\\WHU_Hi_LongKou.mat'))['WHU_Hi_LongKou']
        labels = loadmat(os.path.join(data_path, 'WHU_Hi_LongKou\\WHU_Hi_LongKou_gt.mat'))['WHU_Hi_LongKou_gt']
        class_name = ['Corn', 'Cotton', 'Sesame', 'Broad-leaf soybean','Narrow-leaf soybean', 'Rice', 
                      'Water','Roads and houses', 'Mixed weed'] 
    elif name == 'BO':
        data = loadmat(os.path.join(data_path, 'Botswana\\Botswana.mat'))['Botswana']
        labels = loadmat(os.path.join(data_path, 'Botswana\\Botswana_gt.mat'))['Botswana_gt']
        class_name = ['Water','Hippo grass','Hippo grass 1','Hippo grass 2','Reeds','Riparian','Firescar','Island interior','Acacia woodlands',
                      'Acacia shrublands','Acacia grasslands','Short mopanc','Mixed mopanc','Chalcedony']
    return data, labels, class_name

In [ ]:
data,label,class_name = loadData(DATASET)
NUM_CLASS = label.max()

In [ ]:
def applyPCA(X, numComponents=15):
    """PCA processing

    Args:
        X (ndarray M*N*C): data needs DR
        numComponents (int, optional): number of reserved components. Defaults to 15.

    Returns:
        newX: _description_
    """
    newX = np.reshape(X, (-1, X.shape[2]))
    pca = PCA(n_components=numComponents, whiten=True)   ##PCA and normalization
    newX = pca.fit_transform(newX)
    newX = np.reshape(newX, (X.shape[0],X.shape[1], numComponents))
    return newX

In [ ]:
data = applyPCA(data,N_PCA)
data.shape

In [ ]:
def sample_gt(gt, train_rate, seed):
    """ generate training gt for training dataset
    Args:
        gt (ndarray): full classmap
        train_rate (float): ratio of training dataset
    Returns:
        train_gt(ndarray): classmap of training data
        test_gt(ndarray): classmap of test data
    """
    indices = np.nonzero(gt)  ##([x1,x2,...],[y1,y2,...])
    X = list(zip(*indices))  ## X=[(x1,y1),(x2,y2),...] location of pixels
    y = gt[indices].ravel()
    train_gt = np.zeros_like(gt)
    test_gt = np.zeros_like(gt)
    if train_rate > 1:
       train_rate = int(train_rate)
    train_indices, test_indices = train_test_split(X, train_size=train_rate, stratify=y, random_state=seed)
    train_indices = [t for t in zip(*train_indices)]   ##[[x1,x2,...],[y1,y2,...]]
    test_indices = [t for t in zip(*test_indices)]
    train_gt[tuple(train_indices)] = gt[tuple(train_indices)]
    test_gt[tuple(test_indices)] = gt[tuple(test_indices)]
    
    return train_gt, test_gt

In [ ]:
class PatchSet(Dataset):
    """ Generate 3D patch from hyperspectral dataset """
    def __init__(self, data, gt, patch_size, is_pred=False):
        """
        Args:
            data: 3D hyperspectral image
            gt: 2D array of labels
            patch_size: int, size of the 3D patch
            is_pred: bool, create data without label for prediction (default False) 

        """
        super(PatchSet, self).__init__()
        self.is_pred = is_pred
        self.patch_size = patch_size
        p = self.patch_size // 2
        self.data = np.pad(data,((p,p),(p,p),(0,0)),'constant',constant_values = 0)
        if is_pred:
            gt = np.ones_like(gt)
        self.label = np.pad(gt,(p,p),'constant',constant_values = 0)
        x_pos, y_pos = np.nonzero(gt)
        x_pos, y_pos = x_pos + p, y_pos + p   ##indices after padding
        self.indices = np.array([(x,y) for x,y in zip(x_pos, y_pos)])
        if not is_pred:
            np.random.shuffle(self.indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, y = self.indices[i]
        x1, y1 = x - self.patch_size // 2, y - self.patch_size // 2
        x2, y2 = x1 + self.patch_size, y1 + self.patch_size
        data = self.data[x1:x2, y1:y2]
        label = self.label[x, y]
        data = np.asarray(data, dtype='float32').transpose((2, 0, 1))
        label = np.asarray(label, dtype='int64')
        data = torch.from_numpy(data)
        label = torch.from_numpy(label)
        if self.is_pred:
            return data
        else: return data, label

In [ ]:
class PixelViT(nn.Module):
    def __init__(self, bands):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(bands)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=bands,                            
            nhead=NHEADS,                          
            dim_feedforward=FEEDFORWARD_FACTOR*bands, 
            batch_first=True,                  
            dropout=0.4,
        )
        self.attn = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS) 
        self.layernorm2 = nn.LayerNorm(bands)

    def forward(self, x):
        x = self.layernorm1(x)
        x = self.attn(x)  
        x = self.layernorm2(x)
        out = x.mean(dim=1)  
        return out

In [ ]:
class S3ViT(nn.Module):
    def __init__(self, in_chs, patch_size, class_nums):
        super().__init__()
        self.in_chs = in_chs
        self.patch_size = patch_size
        self.num_blocks = 4

        self.pixel_vit = PixelViT(bands=64)
        self.fc = nn.Linear(64, class_nums) 

    def forward(self, Y):
        # ViT Branch
        B, C, H, W = Y.shape
        y = Y.reshape(B, C, -1).permute(0, 2, 1)  
        vit = self.pixel_vit(y)  
        out = self.fc(vit) 
        return out

In [ ]:
def train(model, train_loader, val_loader, save_path, device):
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), LR, weight_decay=WEIGHT_DECAY)
    loss_func = nn.CrossEntropyLoss()
    train_num = train_loader.dataset.__len__()
    val_num = val_loader.dataset.__len__()

    fig = plt.figure()
    ax1 = fig.add_subplot(2, 1, 1)
    ax2 = fig.add_subplot(2, 1, 2)
    try:
        for e in range(EPOCH):
            model.train()
            avg_loss = 0.
            train_acc = 0
            for batch_idx, (data, target) in enumerate(train_loader):
                data,target = data.to(device),target.to(device)
                optimizer.zero_grad()
                out = model(data)
                target = target - 1  ## class 0 in out is class 1 in target
                loss = loss_func(out,target)
                loss.backward()
                optimizer.step()
                avg_loss += loss.item()
                _,pred = torch.max(out,dim=1)
                train_acc += (pred == target).sum().item()
            loss_list.append(avg_loss/train_num)
            acc_list.append(train_acc/train_num)
            print(f"epoch {e}/{EPOCH} loss:{loss_list[e]}  acc:{acc_list[e]}")
            ## valuation
            if (e+1)%VAL_EPOCH == 0 or (e+1)==EPOCH:
                val_acc =0
                model.eval()
                for batch_idx, (data, target) in enumerate(val_loader):
                    data,target = data.to(device),target.to(device)
                    out = model(data)
                    target = target - 1  ## class 0 in out is class 1 in target
                    _,pred = torch.max(out,dim=1)
                    val_acc += (pred == target).sum().item()
                val_acc_list.append(val_acc/val_num)
                val_epoch_list.append(e)
                print(f"epoch {e}/{EPOCH}  val_acc:{val_acc_list[-1]}")
                save_name = os.path.join(SAVE_PATH, f"epoch_{e}_acc_{val_acc_list[-1]:.4f}.pth")
                torch.save(model.state_dict(),save_name)
        ax1.plot(np.arange(e+1),loss_list)
        ax1.set_title('loss')
        ax1.set_xlabel('epoch')
        ax2.plot(np.arange(e+1),acc_list,label = 'train_acc')
        ax2.plot(val_epoch_list,val_acc_list,label = 'val_acc')
        ax2.set_title('acc')
        ax2.set_xlabel('epoch')
        ax2.legend()
    except Exception as exc:
        print(exc)
    finally: 
        print(f'Stop in epoch {e}')

    return val_acc_list, val_epoch_list

In [ ]:
def test(best_model, label, all_loader, test_gt, device):
    best_model.to(device)
    best_model.eval()
    pred_map = []
    for batch_idx, data in enumerate(all_loader):
        data = data.to(device)
        target = best_model(data)
        _, pred = torch.max(target, dim = 1)
        pred_map += [np.array(pred.detach().cpu() + 1)]   ## class 0 in pred_map is class 1 in gt
    pred_map = np.asarray(np.hstack(pred_map), dtype=np.uint8).reshape(label.shape[0], label.shape[1])

    test_pred = pred_map[test_gt != 0]
    test_true = test_gt[test_gt != 0]

    return test_pred, test_true

In [ ]:
## get best model path and del other models
def get_best_model(acc_list, epoch_list, save_path):
    """get best model path by valuation list

    Args:
        acc_list (list): list of valuation accuracy
        epoch_list (list): list of valuation epoch
        save_path (str): path of save dir

    Returns:
        best_model_path: path of best model
    """
    acc_list = np.array(acc_list)
    epoch_list = np.array(epoch_list)
    best_index = np.argwhere(acc_list==np.max(acc_list))[-1].item()
    best_epoch = epoch_list[best_index]
    best_acc = acc_list[best_index]
    file_name = f"epoch_{best_epoch}_acc_{best_acc:.4f}.pth"
    best_model_path=os.path.join(save_path, file_name)
    print(f"best model:{file_name}")
    ##del save model except best model
    for f in os.listdir(save_path):
        if f[-3:]=='pth' and os.path.join(save_path,f)!=best_model_path:
            os.remove(os.path.join(save_path,f))
    return best_model_path

In [ ]:
# random seeds
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# empty list to storing results
results = []

device = torch.device(DEVICE if DEVICE>=0 and torch.cuda.is_available() else 'cpu')
for run in range(10):
    np.random.seed(seeds[run])
    print("run {} / 10".format(run + 1))
    train_gt, test_gt = sample_gt(label,TRAIN_RATE, seeds[run])
    val_gt, test_gt = sample_gt(test_gt,VAL_RATE/(1-TRAIN_RATE), seeds[run])

    ## display sampling info
    if run == 0:
        sample_report = f"{'class': ^10}{'train_num':^10}{'val_num': ^10}{'test_num': ^10}{'total': ^10}\n"
        for i in np.unique(label):
            if i == 0: continue
            sample_report += f"{i: ^10}{(train_gt==i).sum(): ^10}{(val_gt==i).sum(): ^10}{(test_gt==i).sum(): ^10}{(label==i).sum(): ^10}\n"
        sample_report += f"{'total': ^10}{np.count_nonzero(train_gt): ^10}{np.count_nonzero(val_gt): ^10}{np.count_nonzero(test_gt): ^10}{np.count_nonzero(label): ^10}"
        print(sample_report)

    ##create dataset and dataloader
    train_data = PatchSet(data, train_gt, PATCH_SIZE)
    val_data = PatchSet(data, val_gt, PATCH_SIZE)
    all_data = PatchSet(data, label, PATCH_SIZE,is_pred = True)
    train_loader = DataLoader(train_data,BATCH_SIZE,shuffle= True)
    val_loader = DataLoader(val_data,BATCH_SIZE,shuffle= True)
    all_loader = DataLoader(all_data,BATCH_SIZE,shuffle= False)

    d,g=train_data.__getitem__(0)
    model = S3ViT(d.shape[0],PATCH_SIZE,class_nums=NUM_CLASS)

    if run == 0:
        summary(model, input_size=[(1, d.shape[0], PATCH_SIZE, PATCH_SIZE)],
                col_names=['num_params','kernel_size','mult_adds','input_size','output_size'],
                col_width=10, row_settings=['var_names'], depth=4)

    ## training the model
    val_acc_list, val_epoch_list = train(model, train_loader, val_loader, SAVE_PATH, device)

    ## load best model
    best_model_path = get_best_model(val_acc_list,val_epoch_list,SAVE_PATH)
    best_model = S3ViT(d.shape[0],PATCH_SIZE,class_nums=NUM_CLASS)
    best_model.load_state_dict(torch.load(best_model_path))

    ## inference
    test_pred, test_true = test(best_model, label, all_loader, test_gt, device)

    OA = accuracy_score(test_true,test_pred)
    AA = recall_score(test_true,test_pred,average='macro')
    kappa = cohen_kappa_score(test_true,test_pred)

    # 每个类的准确度
    class_acc = recall_score(
        test_true,
        test_pred,
        average=None,
        labels=np.arange(1, NUM_CLASS + 1)
    )

    # save results for final mean ± std
    results.append({
        "OA": OA,
        "AA": AA,
        "Kappa": kappa,
        "class_acc": class_acc
    })

    del model, best_model, train_data, train_loader, val_data, val_loader, all_data, all_loader

# final summary
oa_list = np.array([r["OA"] for r in results])
aa_list = np.array([r["AA"] for r in results])
kappa_list = np.array([r["Kappa"] for r in results])

class_acc_all = np.stack([r["class_acc"] for r in results], axis=0)
class_acc_mean = np.mean(class_acc_all, axis=0)
class_acc_std = np.std(class_acc_all, axis=0)

final_log = "\n========== Final 10-run Results ==========\n"
final_log += "OA:       {:.4f} ± {:.4f}\n".format(np.mean(oa_list), np.std(oa_list))
final_log += "AA:       {:.4f} ± {:.4f}\n".format(np.mean(aa_list), np.std(aa_list))
final_log += "Kappa:    {:.4f} ± {:.4f}\n".format(np.mean(kappa_list), np.std(kappa_list))

final_log += "\nPer-class accuracy:\n"
for name, mean_acc, std_acc in zip(class_name, class_acc_mean, class_acc_std):
    final_log += "{}: {:.4f} ± {:.4f}\n".format(name, mean_acc, std_acc)

print(final_log)
fp = open(os.path.join(SAVE_PATH, 'final_summary.txt'), 'w+')
fp.writelines(final_log)
fp.close()